In [2]:
import os
import sys
sys.path.append(os.path.join(os.path.dirname(os.getcwd()), 'src'))

from pathlib import Path
import cv2
import pickle
import numpy as np
import matplotlib.pyplot as plt
import math
import json
from datetime import date
plt.rcParams["figure.figsize"] = (24,18)
from ultralytics import YOLO
from utils import *

In [3]:
model_path = os.path.join(os.path.dirname(os.getcwd()), 'models')
model = YOLO(os.path.join(model_path, "yolov8n.pt"))

In [4]:
data_path = os.path.join(os.path.dirname(os.getcwd()), 'data')
vid_name =  'real_test'

In [24]:
from sklearn.cluster import KMeans
from collections import Counter
from colormath.color_objects import sRGBColor, LabColor
from colormath.color_conversions import convert_color
from colormath.color_diff import delta_e_cie2000

images_path = os.path.join(data_path, 'images' + '/' + vid_name)
print(os.path.exists(images_path))
first_batch = os.listdir(images_path)[0:20]
image_batch = [os.path.join(images_path, x) for x in first_batch]        # Get several images to test

team_1_color_global = []
team_2_color_global = []
team_position_global = []

misc_color_global   = []
misc_position_global = []
misc_frequency_global = []

frame_num = 0


for image_ad in image_batch:
    rgb_image = cv2.imread(image_ad, cv2.COLOR_BGR2RGB)

    ## pitch seg
    hsv_image = cv2.cvtColor(rgb_image, cv2.COLOR_BGR2HSV)
    lower_green = np.array([35, 20, 50])
    upper_green = np.array([90, 255, 255])

    mask_green = cv2.inRange(hsv_image, lower_green, upper_green)
    result = cv2.bitwise_and(hsv_image, hsv_image, mask=mask_green)
    temp = cv2.cvtColor(result, cv2.COLOR_HSV2RGB)
    h, w, _ = result.shape
    _, green, _ = cv2.split(temp)

    #ret, thresh = cv2.threshold(temp, 150, 255, cv2.THRESH_BINARY)

    contours, hierarchy = cv2.findContours(image=green, mode=cv2.RETR_EXTERNAL,
                                      method=cv2.CHAIN_APPROX_NONE)

    contours = max(contours, key=cv2.contourArea)
    rect = np.int16(cv2.boundingRect(contours))
    min_x, min_y, w, h = rect
    max_x, max_y = [min_x + w, min_y + h]
    #cv2.polylines(img=temp, pts=[contours], isClosed=False, color=(255, 0, 0), thickness=5)
    cv2.rectangle(temp, (min_x, min_y), (max_x, max_y),color=(255, 0, 0), thickness=5)

    # Crop the image
    result = cv2.resize(result[min_y:max_y, min_x:max_x], (w, h))
    rgb_image = cv2.resize(rgb_image[min_y:max_y, min_x:max_x], (w, h))
    #plt.figure()
    #plt.imshow(rgb_image[:,:,::-1])
    ## run the model

    results = model.predict(rgb_image)
    boxes = results[0].boxes
    # Compute hue of grass
    grass_average = rgb_image[:,:,::-1].mean(axis=0).mean(axis=0)
    # Assignment 
    assignment=[]
    player_x_coord = []
    player_y_coord = []
    crop_img_list = []
    
    for box in boxes:
        if box.cls[0].item() == 0:   # if the person then continue to compute hue
            crop = box.xyxy
            (startX, startY, endX, endY) = np.concatenate(crop.cpu().detach().int().tolist())
            player_saved = rgb_image[startY:endY,startX:endX]
            player_x_coord.append(0.5*(startX + endX))
            player_y_coord.append(0.5*(startY + endY))
            crop_img_list.append(player_saved[:,:,::-1])
            # Z threshold% area of the box into account
            z = 0.6
            new_startY = round(startY + (1-np.sqrt(z))*(endY-startY)/2)
            new_endY = round(endY - (1-np.sqrt(z))*(endY-startY)/2)
            new_startX = round(startX + (1-np.sqrt(z))*(endX-startX)/2)
            new_endX = round(endX - (1-np.sqrt(z))*(endX-startX)/2)
            #--------------------
            cropped = rgb_image[new_startY:new_endY,new_startX:new_endX]
            test_crop = cropped[:,:,::-1].copy()
            result = []
            for row_idx in range(cropped.shape[0]):
                for col_idx in range(cropped.shape[1]):        
                    current_color = test_crop[row_idx][col_idx]
                    if sum(abs(current_color - grass_average)) > 60:
                        result.append(current_color)
            final_color = np.mean(result,axis = 0)
            assignment.append(final_color)

    kmeans = KMeans(n_clusters=3)
    s=kmeans.fit(assignment)
    labels=kmeans.labels_
    teams = Counter(labels).most_common(3)
    team_1 = np.where(labels == teams[0][0])[0]
    team_2 = np.where(labels == teams[1][0])[0]
    misc   = np.where(labels == teams[2][0])[0]


    if frame_num == 0:
        team_1_color = np.mean([assignment[i] for i in team_1],axis =0)
        team_2_color = np.mean([assignment[i] for i in team_2],axis =0)
        print(team_1_color)
        print(team_2_color)
        # Metric CLE: Conversion -> Diff
        srgb_assignment = [sRGBColor(i[0], i[1], i[2]) for i in assignment]
        srgb_team_1_color = sRGBColor(team_1_color[0],team_1_color[1],team_1_color[2])
        srgb_team_2_color = sRGBColor(team_2_color[0],team_2_color[1],team_2_color[2])
        lab_assignment = [convert_color(x,LabColor) for x in srgb_assignment]
        lab_team_1_color = convert_color(srgb_team_1_color, LabColor)
        lab_team_2_color = convert_color(srgb_team_2_color, LabColor)
        team_1_color_global = lab_team_1_color 
        team_2_color_global = lab_team_2_color
        team_1_check = [delta_e_cie2000(lab_team_1_color, lab_assignment[i]) for i in team_1]
        team_2_check = [delta_e_cie2000(lab_team_2_color, lab_assignment[i]) for i in team_2]
        convert_misc_1_idx = [i for i in range(len(team_1_check)) if team_1_check[i] > 50] 
        convert_misc_2_idx = [i for i in range(len(team_2_check)) if team_2_check[i] > 50]

        if len(convert_misc_1_idx) > 0:
            for x in convert_misc_1_idx:
                labels[x] = teams[2][0]
        if len(convert_misc_2_idx) > 0:
            for y in convert_misc_2_idx:
                labels[y] = teams[2][0]
        
        labels = list(map(lambda x: x if x != teams[0][0] else 'Team 1', labels))
        labels = list(map(lambda x: x if x != teams[1][0] else 'Team 2', labels))
        labels = list(map(lambda x: x if x != teams[2][0] else 'Misc', labels))
        print(labels)
        
    
    if frame_num > 0:
        
        team_1_color = np.mean([assignment[i] for i in team_1],axis =0)
        team_2_color = np.mean([assignment[i] for i in team_2],axis =0)                        

        # Repetition to find the misc: Write the correct red
        srgb_assignment = [sRGBColor(i[0], i[1], i[2]) for i in assignment]
        srgb_team_1_color = sRGBColor(team_1_color[0],team_1_color[1],team_1_color[2])
        srgb_team_2_color = sRGBColor(team_2_color[0],team_2_color[1],team_2_color[2])
        lab_assignment = [convert_color(x,LabColor) for x in srgb_assignment]
        lab_team_1_color = convert_color(srgb_team_1_color, LabColor)
        lab_team_2_color = convert_color(srgb_team_2_color, LabColor)
        team_1_check = [delta_e_cie2000(lab_team_1_color, lab_assignment[i]) for i in team_1]
        team_2_check = [delta_e_cie2000(lab_team_2_color, lab_assignment[i]) for i in team_2]
        convert_misc_1_idx = [i for i in range(len(team_1_check)) if team_1_check[i] > 50] 
        convert_misc_2_idx = [i for i in range(len(team_2_check)) if team_2_check[i] > 50]

        if len(convert_misc_1_idx) > 0:
            for x in convert_misc_1_idx:
                labels[x] = teams[2][0]
                #current_misc_color.append(assignment[x])
        if len(convert_misc_2_idx) > 0:
            for y in convert_misc_2_idx:
                labels[y] = teams[2][0]
                #current_misc_color.append(assignment[y])
        

        # Update the label so that team X is always color Y 
        global_1_vs_local_1 = delta_e_cie2000(team_1_color_global, lab_team_1_color)
        global_1_vs_local_2 = delta_e_cie2000(team_1_color_global, lab_team_2_color)
        if  global_1_vs_local_1 < global_1_vs_local_2:
            labels = list(map(lambda x: x if x != teams[0][0] else 'Team 1', labels))
            labels = list(map(lambda x: x if x != teams[1][0] else 'Team 2', labels))
        else: 
            labels = list(map(lambda x: x if x != teams[1][0] else 'Team 1', labels))
            labels = list(map(lambda x: x if x != teams[0][0] else 'Team 2', labels))
        labels = list(map(lambda x: x if x != teams[2][0] else 'Misc', labels)) # Comment this
        print(labels)

    # Positional of the two teams
    new_team_1 = np.where(np.array(labels) == "Team 1")[0]
    new_team_2 = np.where(np.array(labels) == "Team 2")[0]
    new_misc   = np.where(np.array(labels) == "Misc")[0]
    print(new_misc)
    
    avg_x_team_1 = np.mean([player_x_coord[i] for i in new_team_1])
    #avg_y_team_1 = np.mean([player_y_coord[i] for i in new_team_1])
    avg_x_team_2 = np.mean([player_x_coord[i] for i in new_team_2])
    #avg_y_team_2 = np.mean([player_y_coord[i] for i in new_team_2]) 
    if avg_x_team_1 < avg_x_team_2:
        print("Team 1 is on the left")
        left_coord = avg_x_team_1
    else:
        print("Team 2 is on the left")
        left_coord = avg_x_team_2
    
    # Update the misc color: 
    current_misc_color = [assignment[i] for i in new_misc]
    current_misc_pos   = [player_x_coord[i] for i in new_misc]
    for i in range(len(current_misc_color)):
        if current_misc_pos[i] <= left_coord:
            current_misc_pos[i] = "Left"
        else:
            current_misc_pos[i] = "Right"

    # Check availability of the misc clr:
    if len(misc_color_global) == 0:
        for j in range(len(current_misc_color)):
            misc_color_global.append(current_misc_color[j])
            misc_position_global.append(current_misc_pos[j])
            misc_frequency_global.append(1)
    else:
        #global vs local:
        current_misc_srgb = [sRGBColor(i[0], i[1], i[2]) for i in current_misc_color]
        current_misc_lab = [convert_color(i,LabColor) for i in current_misc_srgb]
        global_misc_srgb = [sRGBColor(i[0], i[1], i[2]) for i in misc_color_global]
        global_misc_lab = [convert_color(i,LabColor) for i in global_misc_srgb]
        switch = [False]*len(current_misc_lab)
        for i in range(len(global_misc_lab)):
            for j in range(len(current_misc_lab)):
                misc_diff = delta_e_cie2000(global_misc_lab[i], current_misc_lab[j])
                # If the same shade of color then update the param of existing global, but if it is not then addd into the list
                if misc_diff < 20:
                    if current_misc_pos[j] == misc_position_global[i]:
                        misc_frequency_global[i] += 1
                    else:
                        misc_frequency_global[i] += 1
                        misc_position_global[i] = "Mid"
                    switch[j] = True

        for j in range(len(current_misc_lab)):
            if switch[j] == False:
                if len(misc_color_global) <= 6:
                    misc_color_global.append(current_misc_color[j])
                    misc_position_global.append(current_misc_pos[j])
                    misc_frequency_global.append(1)
                else:
                    if min(misc_frequency_global) == 1:
                        k = misc_frequency_global.index(min(misc_frequency_global))
                        misc_color_global[k] = current_misc_color[j]
                        misc_position_global[k] = current_misc_pos[j]
                        misc_frequency_global[k] = 1

    print(misc_position_global)
    print(misc_frequency_global)
    frame_num +=1

for r in misc_color_global:
    print(r)

print(team_1_color_global)



0: 288x640 17 persons, 1 ball, 6.9ms
Speed: 17.7ms preprocess, 6.9ms inference, 0.9ms postprocess per image at shape (1, 3, 288, 640)


True


/home/huy-ng/Project/Football-analysis/.venv/lib/python3.8/site-packages/sklearn/cluster/_kmeans.py:1412: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)

0: 320x640 10 persons, 1 ball, 7.3ms
Speed: 1.4ms preprocess, 7.3ms inference, 1.1ms postprocess per image at shape (1, 3, 320, 640)


[     173.95      198.95      199.55]
[     216.03       188.1      165.42]
['Misc', 'Team 2', 'Team 2', 'Team 1', 'Team 2', 'Misc', 'Team 1', 'Team 1', 'Team 1', 'Team 2', 'Team 1', 'Team 2', 'Team 1', 'Team 2', 'Team 1', 'Team 2', 'Team 1']
[0 5]
Team 1 is on the left
['Right', 'Right']
[1, 1]


/home/huy-ng/Project/Football-analysis/.venv/lib/python3.8/site-packages/sklearn/cluster/_kmeans.py:1412: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)

0: 288x640 11 persons, 8.0ms
Speed: 1.3ms preprocess, 8.0ms inference, 0.8ms postprocess per image at shape (1, 3, 288, 640)


...

/home/huy-ng/Project/Football-analysis/.venv/lib/python3.8/site-packages/sklearn/cluster/_kmeans.py:1412: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)

0: 320x640 12 persons, 1 ball, 7.1ms
Speed: 1.5ms preprocess, 7.1ms inference, 0.8ms postprocess per image at shape (1, 3, 320, 640)


['Team 2', 'Team 1', 'Team 1', 'Team 1', 'Misc']
[4]
Team 2 is on the left
['Mid', 'Right', 'Mid', 'Mid', 'Mid', 'Mid', 'Right']
[9, 2, 3, 4, 6, 2, 1]


/home/huy-ng/Project/Football-analysis/.venv/lib/python3.8/site-packages/sklearn/cluster/_kmeans.py:1412: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)

0: 352x640 17 persons, 7.0ms
Speed: 1.9ms preprocess, 7.0ms inference, 0.8ms postprocess per image at shape (1, 3, 352, 640)


['Team 2', 'Team 2', 'Team 2', 'Team 2', 'Team 2', 'Team 1', 'Team 1', 'Team 2', 'Misc', 'Team 1', 'Team 1', 'Team 1']
[8]
Team 1 is on the left
['Mid', 'Right', 'Mid', 'Mid', 'Mid', 'Mid', 'Right']
[10, 2, 3, 4, 7, 2, 1]
['Team 2', 'Team 1', 'Team 1', 'Team 2', 'Team 1', 'Team 2', 'Team 2', 'Team 1', 'Team 2', 'Team 2', 'Team 1', 'Team 1', 'Team 2', 'Team 1', 'Team 1', 'Team 1', 'Misc']
[16]
Team 1 is on the left
['Mid', 'Right', 'Mid', 'Mid', 'Mid', 'Mid', 'Right']
[11, 2, 3, 4, 8, 2, 1]
[     82.713      88.943      67.848]
[     171.94      198.66      204.48]
[     187.21      139.05      122.05]
[     222.12      223.64      147.09]
[     110.48      109.16      84.692]
[     185.69      211.44       204.5]
[     118.44      127.96      105.25]
LabColor (lab_l:7505.1685 lab_a:-704.4755 lab_b:-267.9862)


/home/huy-ng/Project/Football-analysis/.venv/lib/python3.8/site-packages/sklearn/cluster/_kmeans.py:1412: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)


In [168]:
labels = np.array(['Team 2', 'Team 2', 'Misc', 'Team 2', 'Misc', 'Team 1', 'Team 1', 'Team 1', 'Team 1', 'Misc'])
team_1_mod = np.where(labels == 'Team 1')
print(team_1_mod)

(array([5, 6, 7, 8]),)


In [10]:
a = []
a.append([3,45,56,34])
a

[[3, 45, 56, 34]]

In [58]:
!pip install colormath

  ERROR: Command errored out with exit status 1:
   command: /home/huy-ng/Project/Football-analysis/.venv/bin/python3 -u -c 'import sys, setuptools, tokenize; sys.argv[0] = '"'"'/tmp/pip-install-48t9a4v9/colormath/setup.py'"'"'; __file__='"'"'/tmp/pip-install-48t9a4v9/colormath/setup.py'"'"';f=getattr(tokenize, '"'"'open'"'"', open)(__file__);code=f.read().replace('"'"'\r\n'"'"', '"'"'\n'"'"');f.close();exec(compile(code, __file__, '"'"'exec'"'"'))' bdist_wheel -d /tmp/pip-wheel-z0sq_de6
       cwd: /tmp/pip-install-48t9a4v9/colormath/
  Complete output (6 lines):
  usage: setup.py [global_opts] cmd1 [cmd1_opts] [cmd2 [cmd2_opts] ...]
     or: setup.py --help [cmd1 cmd2 ...]
     or: setup.py --help-commands
     or: setup.py cmd --help
  
  error: invalid command 'bdist_wheel'
  ----------------------------------------
  ERROR: Failed building wheel for colormath
  Running setup.py clean for colormath
Failed to build colormath
    Running setup.py install for colormath ... done


In [71]:
# Red Color
color1_rgb = sRGBColor(1.0, 0.0, 0.0);

# Blue Color
color2_rgb = sRGBColor(0.0, 0.0, 1.0);

# Convert from RGB to Lab Color Space
color1_lab = convert_color(color1_rgb, LabColor);

# Convert from RGB to Lab Color Space
color2_lab = convert_color(color2_rgb, LabColor);

# Find the color difference
delta_e = delta_e_cie2000(color1_lab, color2_lab);

print ("The difference between the 2 color = ", delta_e)

The difference between the 2 color =  52.88009898346556


In [79]:
test_1 = [5.515201631567911, 15.198703983638413, 20.571191867856946, 45.50415708798051, 5.714887319599944, 26.893941278762274, 16.01699691061821, 9.819217559457474, 10.480740217970085]
test_2 = [15.208873593682302, 5.123934739907209, 20.138532797062926, 30.015656082601115, 26.61448881207698, 72.868181894459, 45.00607734138506]
print(np.mean(test_1) + np.std(test_1))
print(np.mean(test_2) + np.std(test_2))

29.22662658034855
51.43221419215469


In [6]:
import numpy

def patch_asscalar(a):
    return a.item()

setattr(numpy, "asscalar", patch_asscalar)

In [12]:
a = [23,43,234,54,12,87,12,87]
a.index(min(a))

4